In [ ]:

# ==== Colab setup (run first) ====
!pip -q install "transformers>=4.48,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" pillow pandas

import torch
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", torch.cuda.get_device_properties(0).total_memory/1024**3, "GB")


# IDEFICS2-8B for VS Code / Jupyter (fixed lightweight version)

이 버전은 네 desktop baseline 구조를 참고해서 **VS Code / 로컬 Jupyter**에서 돌아가도록 다시 정리한 버전이다.

기존 안 돌아가던 원인 쪽에서 크게 바꾼 점:
- `google.colab` 제거
- **`score` 방식 제거** → **`generate` 방식만 사용**
- 경로를 로컬 기준으로 직접 설정
- 단계별 로그 출력 추가
- 전체 추론 전에 **1개 샘플 smoke test**를 먼저 돌리도록 구성
- 파싱 실패 시 기본값으로 넘기지 않고 에러 발생
- 저장 경로를 로컬 파일로 수정

권장 순서:
1. 경로 설정
2. 데이터 확인
3. 모델 로드
4. smoke test
5. quick eval
6. full inference


In [ ]:
# 필요 시 1회 설치
# 이미 환경이 맞으면 건너뛰어도 됨
%pip install -U "transformers>=4.48,<4.58" "accelerate>=0.34.2" "bitsandbytes>=0.43.3" pandas pillow tqdm

In [ ]:

from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig
import torch

MODEL_ID = "HuggingFaceM4/idefics2-8b"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

if hasattr(processor, "image_processor"):
    processor.image_processor.do_image_splitting = False

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

print("model loaded")


## 경로 설정

In [ ]:
# 네 환경에 맞게 여기만 수정
DATA_ROOT = Path(".").resolve()   # 예: Path(r"D:\ssafy_vqa")
ZIP_PATH = DATA_ROOT / "2026-ssafy-15-2-ai.zip"
EXTRACT_DIR = DATA_ROOT

TRAIN_CSV = EXTRACT_DIR / "train.csv"
TEST_CSV = EXTRACT_DIR / "test.csv"

print("DATA_ROOT :", DATA_ROOT)
print("ZIP_PATH  :", ZIP_PATH)
print("EXTRACT   :", EXTRACT_DIR)

In [ ]:
# zip 파일이 있고 아직 csv가 없으면 압축 해제
if ZIP_PATH.exists() and not TRAIN_CSV.exists():
    print(f"[INFO] extracting {ZIP_PATH} ...")
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"[INFO] extraction done in {time.time()-t0:.1f}s")
else:
    print("[INFO] skip extraction")
    print("ZIP exists     :", ZIP_PATH.exists())
    print("train.csv exists:", TRAIN_CSV.exists())

## 기본 설정 / 데이터 로드

In [ ]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

MODEL_ID = "HuggingFaceM4/idefics2-8b"

# 무거운 score 방식은 제거하고 generate만 사용
MAX_NEW_TOKENS = 2
DO_IMAGE_SPLITTING = False

# 빠른 테스트용
QUICK_EVAL_N = 10
TEST_LIMIT = None   # 예: 100 으로 넣으면 test 100개만 먼저 실행

if not TRAIN_CSV.exists():
    raise FileNotFoundError(f"train.csv not found: {TRAIN_CSV}")
if not TEST_CSV.exists():
    raise FileNotFoundError(f"test.csv not found: {TEST_CSV}")

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

required_train_cols = {"path", "question", "a", "b", "c", "d", "answer"}
required_test_cols = {"id", "path", "question", "a", "b", "c", "d"}

missing_train = required_train_cols - set(train_df.columns)
missing_test = required_test_cols - set(test_df.columns)

if missing_train:
    raise ValueError(f"train.csv missing columns: {sorted(missing_train)}")
if missing_test:
    raise ValueError(f"test.csv missing columns: {sorted(missing_test)}")

def resolve_path(p: str) -> str:
    p = str(p)
    path_obj = Path(p)
    if path_obj.is_absolute():
        return str(path_obj)
    return str((EXTRACT_DIR / path_obj).resolve())

train_df["path"] = train_df["path"].map(resolve_path)
test_df["path"] = test_df["path"].map(resolve_path)

if TEST_LIMIT is not None:
    test_df = test_df.iloc[:TEST_LIMIT].reset_index(drop=True)

print("train shape:", train_df.shape)
print("test shape :", test_df.shape)
display(train_df.head(2))
display(test_df.head(2))

## 모델 로드

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU가 필요합니다. 현재 CUDA를 사용할 수 없습니다.")

print("[INFO] loading processor...")
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f"[INFO] processor loaded in {time.time()-t0:.1f}s")

if hasattr(processor, "image_processor") and hasattr(processor.image_processor, "do_image_splitting"):
    processor.image_processor.do_image_splitting = DO_IMAGE_SPLITTING
    print("[INFO] do_image_splitting =", processor.image_processor.do_image_splitting)

print("[INFO] loading 4bit model... (첫 실행은 다운로드 때문에 오래 걸릴 수 있음)")
t0 = time.time()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()

print(f"[INFO] model loaded in {time.time()-t0:.1f}s")
print("[INFO] model dtype:", next(model.parameters()).dtype)

## 프롬프트

In [ ]:
SYSTEM_INSTRUCT = (
    "You are an expert multiple-choice visual question answering model. "
    "Use the image as the primary evidence. "
    "Carefully inspect text, numbers, symbols, colors, counts, positions, objects, and relationships. "
    "Compare all four options and choose the single best answer supported by the image. "
    "Return exactly one lowercase letter: a, b, c, or d."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{SYSTEM_INSTRUCT}\n\n"
        "Question:\n"
        f"{question}\n\n"
        "Options:\n"
        f"a) {a}\n"
        f"b) {b}\n"
        f"c) {c}\n"
        f"d) {d}\n\n"
        "Answer with exactly one lowercase letter only."
    )

CHOICES = ["a", "b", "c", "d"]

## 유틸 함수

## Training (Colab safe)

In [ ]:

from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass

class VQADataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = load_image(row["path"])
        prompt = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

        messages = [
            {"role":"user","content":[{"type":"image"},{"type":"text","text":prompt}]},
            {"role":"assistant","content":[{"type":"text","text":row["answer"]}]}
        ]
        return {"messages":messages,"image":img}

@dataclass
class Collator:
    processor: any

    def __call__(self, batch):
        texts, images = [], []
        for b in batch:
            text = processor.apply_chat_template(b["messages"], tokenize=False)
            texts.append(text)
            images.append(b["image"])

        enc = processor(text=texts, images=images, return_tensors="pt", padding=True)
        labels = enc["input_ids"].clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        enc["labels"] = labels
        return enc


In [ ]:

# reduce size for stability
train_df_small = train_df.sample(n=min(50, len(train_df)), random_state=42)

train_loader = DataLoader(
    VQADataset(train_df_small),
    batch_size=1,
    shuffle=True,
    collate_fn=Collator(processor)
)


In [ ]:

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

model.train()

for epoch in range(1):
    for batch in tqdm(train_loader):
        batch = {k:v.to(model.device) for k,v in batch.items()}
        loss = model(**batch).loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

print("training done")


In [ ]:
def load_image(image_path: str) -> Image.Image:
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")
    return Image.open(image_path).convert("RGB")

def build_messages(row):
    prompt = build_mc_prompt(
        str(row["question"]),
        str(row["a"]),
        str(row["b"]),
        str(row["c"]),
        str(row["d"]),
    )
    return [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]

def extract_choice(text: str) -> str:
    if not isinstance(text, str):
        raise TypeError(f"Model output is not str: {type(text)}")

    text = text.strip().lower()
    m = re.search(r"\b([abcd])\b", text)
    if m:
        return m.group(1)

    compact = text.replace(" ", "").replace("\n", "")
    if compact and compact[0] in CHOICES:
        return compact[0]

    raise ValueError(f"Failed to parse choice from output: {text!r}")

def encode_example(row):
    image = load_image(str(row["path"]))
    messages = build_messages(row)
    prompt_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = processor(
        text=[prompt_text],
        images=[image],
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    return image, prompt_text, inputs

@torch.no_grad()
def predict_row(row):
    image, prompt_text, inputs = encode_example(row)

    generated = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=True,
        eos_token_id=processor.tokenizer.eos_token_id,
    )

    new_tokens = generated[:, inputs["input_ids"].shape[1]:]
    decoded = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
    pred = extract_choice(decoded)
    return pred, decoded

## 1개 샘플 smoke test

In [ ]:
sample_row = train_df.iloc[0]

t0 = time.time()
pred, raw = predict_row(sample_row)
dt = time.time() - t0

print("question:", sample_row["question"])
print("gold    :", sample_row["answer"])
print("pred    :", pred)
print("raw     :", repr(raw))
print(f"time     : {dt:.2f}s")

## Quick Eval

In [ ]:
sample_eval_df = train_df.sample(
    n=min(QUICK_EVAL_N, len(train_df)),
    random_state=SEED
).reset_index(drop=True)

records = []
correct = 0

for i in tqdm(range(len(sample_eval_df)), desc="Quick Eval", unit="sample"):
    row = sample_eval_df.iloc[i]
    pred, raw = predict_row(row)
    gold = str(row["answer"]).strip().lower()
    is_correct = int(pred == gold)
    correct += is_correct
    records.append({
        "question": row["question"],
        "gold": gold,
        "pred": pred,
        "correct": is_correct,
        "raw": raw,
    })

quick_eval_df = pd.DataFrame(records)
quick_acc = correct / len(sample_eval_df) if len(sample_eval_df) else 0.0
print(f"Quick Eval Accuracy: {quick_acc:.4f}")
display(quick_eval_df.head(10))

## 전체 추론

In [ ]:
preds = []
raw_outputs = []

for i in tqdm(range(len(test_df)), desc="Full Inference", unit="sample"):
    row = test_df.iloc[i]
    pred, raw = predict_row(row)
    preds.append(pred)
    raw_outputs.append(raw)

submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": preds,
})

submission_path = EXTRACT_DIR / "submission_idefics2_8b_vs_fixed.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
display(submission.head())

## 예측 분포

In [ ]:
pd.Series(preds).value_counts(dropna=False).sort_index()